In [11]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling2D, Dense, Dropout, Multiply, Reshape
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
import joblib

# ------------------ Paths ------------------
DATASET_PATH = "/Users/koushal/Desktop/desktop/ml2 project/dataset_audio"

# ------------------ Parameters ------------------
SAMPLE_RATE = 22050
DURATION = 3
N_MELS = 128
HOP_LENGTH = 512
MAX_LEN = 128

# ------------------ Feature Extraction ------------------
def extract_mel_spectrogram(file_path, max_len=MAX_LEN):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, res_type="kaiser_fast")
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    if mel_db.shape[1] < max_len:
        pad_width = max_len - mel_db.shape[1]
        mel_db = np.pad(mel_db, pad_width=((0,0), (0,pad_width)), mode="constant")
    else:
        mel_db = mel_db[:, :max_len]

    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())
    return mel_db

# ------------------ Load Dataset ------------------
def load_dataset(split="train"):
    X, y = [], []
    split_path = os.path.join(DATASET_PATH, split)

    for label in os.listdir(split_path):
        label_path = os.path.join(split_path, label)
        if not os.path.isdir(label_path):
            continue
        for file in os.listdir(label_path):
            if file.endswith(".wav"):
                file_path = os.path.join(label_path, file)
                mel = extract_mel_spectrogram(file_path)
                X.append(mel)
                y.append(label)

    return np.array(X), np.array(y)

print("Loading training data...")
X_train, y_train = load_dataset("train")
print("Loading validation data...")
X_val, y_val = load_dataset("val")
print("Loading test data...")
X_test, y_test = load_dataset("test")

# ------------------ Encode Labels ------------------
encoder = LabelEncoder()
y_train_enc = to_categorical(encoder.fit_transform(y_train))
y_val_enc = to_categorical(encoder.transform(y_val))
y_test_enc = to_categorical(encoder.transform(y_test))

# ------------------ Reshape for CNN ------------------
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"✅ Train shape: {X_train.shape}, Labels: {y_train_enc.shape}")
print(f"Classes: {encoder.classes_}")

# ------------------ Residual Block with SE ------------------
def se_block(input_tensor, reduction=16):
    channels = int(input_tensor.shape[-1])
    se = GlobalAveragePooling2D()(input_tensor)
    se = Dense(channels // reduction, activation="relu")(se)
    se = Dense(channels, activation="sigmoid")(se)
    se = Reshape((1,1,channels))(se)
    return Multiply()([input_tensor, se])

def residual_block(x, filters, stride=1):
    shortcut = x
    x = Conv2D(filters, (3,3), strides=stride, padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, (3,3), strides=1, padding="same")(x)
    x = BatchNormalization()(x)

    x = se_block(x)  # squeeze-and-excitation

    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = Conv2D(filters, (1,1), strides=stride, padding="same")(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

# ------------------ Improved CNN Model ------------------
def build_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)

    x = residual_block(inputs, 32, stride=2)
    x = residual_block(x, 64, stride=2)
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 256, stride=2)

    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    return model

model = build_model((N_MELS, MAX_LEN, 1), num_classes=len(encoder.classes_))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# ------------------ Callbacks ------------------
checkpoint = ModelCheckpoint("best_audio_model.keras", monitor="val_accuracy", save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1)

# ------------------ Train ------------------
history = model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=20,
    batch_size=16,
    callbacks=[checkpoint, early_stop, lr_scheduler]
)

# ------------------ Evaluate ------------------
test_loss, test_acc = model.evaluate(X_test, y_test_enc, verbose=1)
print(f"🎯 Test Accuracy: {test_acc:.2f}")

# ------------------ Save Encoder ------------------
joblib.dump(encoder, "label_encoder.pkl")
print("✅ Model and encoder saved!")

Loading training data...
Loading validation data...
Loading test data...


/var/folders/wv/p6lwly3j0tn204_ppsn0wmhr0000gn/T/ipykernel_2508/4044528729.py:39: RuntimeWarning: invalid value encountered in divide
  mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())


✅ Train shape: (2612, 128, 128, 1), Labels: (2612, 3)
Classes: ['happy' 'neutral' 'sad']
Epoch 1/20


2025-09-01 22:11:47.534933: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - accuracy: 0.5467 - loss: 0.9602
Epoch 1: val_accuracy improved from -inf to 0.33333, saving model to best_audio_model.keras
164/164 ━━━━━━━━━━━━━━━━━━━━ 29s 137ms/step - accuracy: 0.5470 - loss: 0.9597 - val_accuracy: 0.3333 - val_loss: 16.1207 - learning_rate: 0.0010
Epoch 2/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.6789 - loss: 0.7610
Epoch 2: val_accuracy improved from 0.33333 to 0.33642, saving model to best_audio_model.keras
164/164 ━━━━━━━━━━━━━━━━━━━━ 20s 124ms/step - accuracy: 0.6789 - loss: 0.7610 - val_accuracy: 0.3364 - val_loss: 2.9637 - learning_rate: 0.0010
Epoch 3/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.6872 - loss: 0.7640
Epoch 3: val_accuracy improved from 0.33642 to 0.47222, saving model to best_audio_model.keras
164/164 ━━━━━━━━━━━━━━━━━━━━ 21s 127ms/step - accuracy: 0.6873 - loss: 0.7638 - val_accuracy: 0.4722 - val_loss: 1.4891 - learning_rate: 0.0010
Epoch 4/20
164/164 ━━━━━━━━━━━━━

In [10]:
import resampy
import librosa
print("✅ Both working fine now")

✅ Both working fine now
